# Calidad de datos de Instacart

**Lección 1: Exploración y manipulación de DataFrames**

Este cuaderno explora las cinco tablas **sin modificarlas** y produce la lista de problemas de calidad que se resuelven en la Lección 2 (`pandas_labs.ipynb`). El resumen de hallazgos está en `reports/data_quality.md`.

## Preparación

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
print(pd.__version__)

3.0.6


## Carga de archivos y el parámetro `sep`

In [2]:
prueba = pd.read_csv(RUTA / 'aisles.csv')
print(prueba.shape)
print(prueba.columns.tolist())
# Esperado: (134, 1) ['aisle_id;aisle'] -> sin sep=';' todo queda en una sola columna

(134, 1)
['aisle_id;aisle']


In [3]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [4]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)
# Esperado: orders (478967, 6) / order_prods (4545007, 4) / products (49694, 4) / aisles (134, 2) / departments (21, 2)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


## Vista rápida con `head`, `sample` e `info`

In [5]:
orders.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


In [6]:
orders.sample(5, random_state=42)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478011,187515,84759,1,2,15,NaN
176737,14154,184924,41,2,15,3.0
402189,1978043,151955,21,1,23,7.0
111658,2907266,43250,11,1,8,1.0
59935,3245658,115414,5,2,10,7.0


In [7]:
orders.info()
# Esperado: cinco columnas int64 y days_since_prior_order float64 con 450148 no nulos

<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB


## Nulos y duplicados con `isna` y `duplicated`

In [8]:
print(orders.isna().sum(), '\n')
print(products.isna().sum(), '\n')
print(order_prods.isna().sum(), '\n')

print(orders.duplicated().sum())
print(orders['order_id'].is_unique)
# Esperado: 28819 / 1258 / 836 nulos, 15 duplicados y False

order_id                      0
user_id                       0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order    28819
dtype: int64 

product_id          0
product_name     1258
aisle_id            0
department_id       0
dtype: int64 

order_id               0
product_id             0
add_to_cart_order    836
reordered              0
dtype: int64 

15
False


## Rangos y categorías con `describe` y `value_counts`

In [9]:
orders.describe()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,4.789670e+05,478967.000000,478967.000000,478967.000000,478967.000000,450148.000000
mean,1.709947e+06,102966.537475,17.157808,2.775058,13.446676,11.101751
std,9.869480e+05,59525.328991,17.704688,2.045869,4.224986,9.189975
min,4.000000e+00,2.000000,1.000000,0.000000,0.000000,0.000000
25%,8.564685e+05,51414.000000,5.000000,1.000000,10.000000,4.000000
50%,1.709522e+06,102633.000000,11.000000,3.000000,13.000000,7.000000
75%,2.562990e+06,154419.000000,23.000000,5.000000,16.000000,15.000000
max,3.421079e+06,206209.000000,100.000000,6.000000,23.000000,30.000000


In [10]:
orders['order_dow'].value_counts().sort_index()
# Esperado: el día 0 (domingo) tiene 84090 pedidos

order_dow
0    84090
1    82185
2    65833
3    60912
4    59810
5    63488
6    62649
Name: count, dtype: int64

In [11]:
departments.tail(3)
# Esperado: la última fila es 21 missing (valor centinela)

,department_id,department
18,19,snacks
19,20,deli
20,21,missing


## Selección con `loc` e `iloc` y columnas nuevas

In [12]:
practica = orders.copy()

practica[['order_id', 'user_id']].head(3)

,order_id,user_id
0,1515936,183418
1,1690866,163593
2,1454967,39980


In [13]:
practica.iloc[0:3, 0:2]

,order_id,user_id
0,1515936,183418
1,1690866,163593
2,1454967,39980


In [14]:
primeros = practica.loc[practica['order_number'] == 1, ['order_id', 'user_id']]
print(len(primeros))
primeros.head()
# Esperado: 28819 filas

28819


,order_id,user_id
28,133707,182261
96,787445,25685
100,294410,111449
103,2869915,123958
104,2521921,42286


In [15]:
practica['es_primer_pedido'] = practica['order_number'] == 1
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
practica['dia'] = practica['order_dow'].map(dias)
print(orders.shape)
practica.head()
# Esperado: (478967, 6) y dos columnas nuevas en practica

(478967, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,es_primer_pedido,dia
0,1515936,183418,11,6,13,30.0,False,sábado
1,1690866,163593,5,5,12,9.0,False,viernes
2,1454967,39980,4,5,19,2.0,False,viernes
3,1768857,82516,56,0,20,10.0,False,domingo
4,3007858,196724,2,4,12,17.0,False,jueves


**Problemas encontrados para la Lección 2:** separador `;`, 15 pedidos duplicados, nulos en tres columnas, tipos `float64` en columnas enteras y la categoría centinela `missing`.

# Laboratorio guiado 1: ficha técnica de los datos

Para cada tabla: cuántas filas tiene, qué tipos y nulos hay en cada columna, si su clave es única y si las tablas encajan entre sí.

### Paso 1: ficha de una tabla

In [16]:
def ficha(df):
    return pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'nulos': df.isna().sum(),
        'pct_nulos': (df.isna().mean() * 100).round(2),
        'distintos': df.nunique(),
        'ejemplo': df.iloc[0],
    })

ficha(orders)
# Esperado: order_id con 478952 distintos (de 478967 filas) y days_since_prior_order con 28819 nulos (6,02 %)

,tipo,nulos,pct_nulos,distintos,ejemplo
order_id,int64,0,0.00,478952,1515936.0
user_id,int64,0,0.00,157437,183418.0
order_number,int64,0,0.00,100,11.0
order_dow,int64,0,0.00,7,6.0
order_hour_of_day,int64,0,0.00,24,13.0
days_since_prior_order,float64,28819,6.02,31,30.0


### Paso 2: reutilizar la función

In [17]:
ficha(products)
# Esperado: product_name tipo str, 1258 nulos (2,53 %) y 48436 nombres distintos

,tipo,nulos,pct_nulos,distintos,ejemplo
product_id,int64,0,0.00,49694,1
product_name,str,1258,2.53,48436,Chocolate Sandwich Cookies
aisle_id,int64,0,0.00,134,61
department_id,int64,0,0.00,21,19


### Paso 3: inventario de las cinco tablas

In [18]:
tablas = {'orders': orders, 'order_prods': order_prods, 'products': products,
          'aisles': aisles, 'departments': departments}

inventario = pd.DataFrame([
    {'tabla': nombre,
     'filas': len(df),
     'columnas': df.shape[1],
     'nulos': int(df.isna().sum().sum()),
     'duplicados': int(df.duplicated().sum()),
     'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1)}
    for nombre, df in tablas.items()
])
inventario

,tabla,filas,columnas,nulos,duplicados,memoria_mb
0,orders,478967,6,28819,15,23.0
1,order_prods,4545007,4,836,0,145.4
2,products,49694,4,1258,0,3.1
3,aisles,134,2,0,0,0.0
4,departments,21,2,0,0,0.0


### Paso 4: comprobar las claves

In [19]:
claves = {'orders': 'order_id', 'products': 'product_id',
          'aisles': 'aisle_id', 'departments': 'department_id'}

for tabla, clave in claves.items():
    print(tabla, clave, tablas[tabla][clave].is_unique)
# Esperado: False solo para orders

orders order_id False
products product_id True
aisles aisle_id True
departments department_id True


### Paso 5: comprobar que las tablas encajan

In [20]:
relaciones = [
    ('order_prods', 'order_id',      'orders',      'order_id'),
    ('order_prods', 'product_id',    'products',    'product_id'),
    ('products',    'aisle_id',      'aisles',      'aisle_id'),
    ('products',    'department_id', 'departments', 'department_id'),
]

for hija, col_hija, padre, col_padre in relaciones:
    huerfanos = (~tablas[hija][col_hija].isin(tablas[padre][col_padre])).sum()
    print(f'{hija}.{col_hija} -> {padre}.{col_padre}: {huerfanos} huérfanos')

print('Pedidos sin productos:', (~orders['order_id'].isin(order_prods['order_id'])).sum())
# Esperado: 0 huérfanos en las cuatro relaciones y Pedidos sin productos: 28906

order_prods.order_id -> orders.order_id: 0 huérfanos
order_prods.product_id -> products.product_id: 0 huérfanos
products.aisle_id -> aisles.aisle_id: 0 huérfanos
products.department_id -> departments.department_id: 0 huérfanos
Pedidos sin productos: 28906


### Paso 6: registrar la bitácora como tabla

In [21]:
bitacora = pd.DataFrame([
    {'tabla': 'orders', 'problema': 'Filas duplicadas', 'evidencia': 15},
    {'tabla': 'orders', 'problema': 'Nulos en days_since_prior_order', 'evidencia': 28819},
    {'tabla': 'products', 'problema': 'Nulos en product_name', 'evidencia': 1258},
    {'tabla': 'order_prods', 'problema': 'Nulos en add_to_cart_order', 'evidencia': 836},
    {'tabla': 'orders', 'problema': 'Pedidos sin productos', 'evidencia': 28906},
])
bitacora.to_csv(SALIDA / 'data_quality_log.csv', index=False)
bitacora

,tabla,problema,evidencia
0,orders,Filas duplicadas,15
1,orders,Nulos en days_since_prior_order,28819
2,products,Nulos en product_name,1258
3,order_prods,Nulos en add_to_cart_order,836
4,orders,Pedidos sin productos,28906


### Conclusiones del laboratorio

- La función `ficha` hace la exploración repetible: se escribe una vez y sirve para cualquier tabla.
- Comparar `nunique()` de la clave con el número de filas revela los duplicados sin buscarlos.
- Las cuatro relaciones no tienen huérfanos, pero hay 28 906 pedidos sin detalle de productos: se tendrá en cuenta al combinar tablas en la Lección 4.
- La bitácora es el puente entre explorar y limpiar: cada fila es una tarea de la Lección 2. Se guarda en `data/processed/data_quality_log.csv`.

# Reto 1

### 1. Productos distintos en `order_prods` y productos nunca pedidos

In [22]:
pedidos_distintos_prod = order_prods['product_id'].nunique()
nunca_pedidos = (~products['product_id'].isin(order_prods['product_id'])).sum()
print('Productos distintos en order_prods:', pedidos_distintos_prod)
print('Productos del catálogo nunca pedidos:', nunca_pedidos)

Productos distintos en order_prods: 45573
Productos del catálogo nunca pedidos: 4121


**Respuesta:** _(completar)_

### 2. Pedidos distintos en `order_prods` frente a las filas de `orders`

In [23]:
pedidos_con_detalle = order_prods['order_id'].nunique()
print('Pedidos distintos en order_prods:', pedidos_con_detalle)
print('Filas de orders:', len(orders))
print('Diferencia:', len(orders) - pedidos_con_detalle)

Pedidos distintos en order_prods: 450046
Filas de orders: 478967
Diferencia: 28921


**Respuesta:** _(completar: ¿qué hace sospechar la diferencia? Relaciónala con los duplicados y con los pedidos sin productos del Paso 5)_

### 3. Máximo y mediana de `add_to_cart_order`

In [24]:
print('Máximo:', order_prods['add_to_cart_order'].max())
print('Mediana:', order_prods['add_to_cart_order'].median())

Máximo: 64.0
Mediana: 6.0


**Respuesta:** _(completar)_

### 4. Porcentaje de recompras

In [25]:
pct_recompras = (order_prods['reordered'] == 1).mean() * 100
print(f'Recompras: {pct_recompras:.2f} % de las filas de order_prods')

Recompras: 59.05 % de las filas de order_prods


**Respuesta:** _(completar)_

### 5. Departamento con más productos en el catálogo

In [26]:
conteo_depto = products['department_id'].value_counts()
id_top = conteo_depto.idxmax()
nombre_top = departments.loc[departments['department_id'] == id_top, 'department'].iloc[0]
print(f'Departamento {id_top} ({nombre_top}): {conteo_depto.max()} productos')

Departamento 11 (personal care): 6564 productos


**Respuesta:** _(completar)_

### 6. Columna `tipo_cliente` en una copia de `orders`

In [27]:
copia = orders.copy()
copia['tipo_cliente'] = np.where(copia['order_number'] == 1, 'nuevo', 'recurrente')
print(copia['tipo_cliente'].value_counts())
print(orders.shape)   # orders no cambia

tipo_cliente
recurrente    450148
nuevo          28819
Name: count, dtype: int64
(478967, 6)


**Respuesta:** _(completar)_

### 7. Filas 100 a 104 de `products` con `iloc`

In [28]:
filas = products.iloc[100:105]
print(filas['product_id'].tolist())
filas

[101, 102, 103, 104, 105]


,product_id,product_name,aisle_id,department_id
100,101,Bread- Healthy Whole Grain,112,3
101,102,Danish Butter Cookies,61,19
102,103,Sprouted Kale Cracker,78,19
103,104,High Fiber Tomato Basil Soup,69,15
104,105,"Easy Grab 9\""x13\"" Oblong Glass Bakeware",10,17


**Respuesta:** _(completar: recuerda que `iloc[100:105]` incluye la posición 100 y excluye la 105)_